# Phase 1 — Exploration des données

**Objectif de ce notebook :**
Charger, nettoyer et explorer les données BTC/ETH.
À la fin, on doit pouvoir répondre à :
1. Les données sont-elles propres et bien alignées ?
2. Quelle est la distribution des log-returns ?
3. La corrélation BTC/ETH est-elle stable dans le temps ?

---

In [ ]:
import sys
sys.path.append('..')  # Pour importer depuis src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.processing.cleaning import load_ohlcv, align_pairs, detect_gaps
from src.features.features import log_returns, rolling_zscore, rolling_correlation
from src.utils.plotting import plot_prices, plot_log_returns, plot_rolling_correlation

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

## 1. Chargement et vérification des données

In [ ]:
df_btc = load_ohlcv('BTCUSDT', '15m')
df_eth = load_ohlcv('ETHUSDT', '15m')

print('\nBTC — premières lignes :')
df_btc.head(3)

In [ ]:
# Vérification des gaps
print('=== Gaps BTC ===')
gaps_btc = detect_gaps(df_btc, timeframe_minutes=15)

print('\n=== Gaps ETH ===')
gaps_eth = detect_gaps(df_eth, timeframe_minutes=15)

In [ ]:
# Alignement
df_btc, df_eth = align_pairs(df_btc, df_eth, name_a='BTC', name_b='ETH')
print(f'\nDataset final : {len(df_btc):,} bougies alignées')

## 2. Visualisation des prix bruts

In [ ]:
plot_prices(df_btc, df_eth, name_a='BTC', name_b='ETH')

## 3. Log-returns

Rappel de la formule : $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$

On attend des log-returns :
- Centrés autour de 0
- Distribution proche d'une gaussienne (avec des queues épaisses en finance)
- Stationnarité visuelle (pas de tendance)

In [ ]:
df_btc['log_return'] = log_returns(df_btc['Close'])
df_eth['log_return'] = log_returns(df_eth['Close'])

# On drop le premier NaN
ret_btc = df_btc['log_return'].dropna()
ret_eth = df_eth['log_return'].dropna()

plot_log_returns(ret_btc, ret_eth)

In [ ]:
# Statistiques descriptives
stats = pd.DataFrame({
    'BTC': ret_btc.describe(),
    'ETH': ret_eth.describe()
})

# Ajouter skewness et kurtosis
stats.loc['skewness'] = [ret_btc.skew(), ret_eth.skew()]
stats.loc['kurtosis'] = [ret_btc.kurtosis(), ret_eth.kurtosis()]

print('Statistiques des log-returns :')
print(stats.round(6))
print('\nNote : kurtosis > 0 = queues plus épaisses qu\'une gaussienne (leptokurtique)')
print('C\'est normal et attendu en finance — les événements extrêmes sont plus fréquents')

In [ ]:
# Distribution des log-returns vs gaussienne
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ret, name in zip(axes, [ret_btc, ret_eth], ['BTC', 'ETH']):
    sns.histplot(ret, bins=200, kde=True, ax=ax, color='steelblue', alpha=0.6)
    
    # Gaussienne théorique
    x = np.linspace(ret.min(), ret.max(), 300)
    from scipy import stats as scipy_stats
    gauss = scipy_stats.norm.pdf(x, ret.mean(), ret.std())
    ax.plot(x, gauss * len(ret) * (ret.max() - ret.min()) / 200,
            color='red', linewidth=1.5, linestyle='--', label='Gaussienne théorique')
    
    ax.set_xlim(ret.quantile(0.001), ret.quantile(0.999))
    ax.set_title(f'Distribution log-returns {name}')
    ax.legend()

plt.tight_layout()
plt.show()
print('→ Les queues sont plus épaisses que la gaussienne : les crashes/rallyes sont plus fréquents que prévu')

## 4. Corrélation mobile BTC/ETH

Si la corrélation est haute et stable → bonne candidate pour le pair trading.
Si elle s'effondre régulièrement → la paire est moins fiable.

In [ ]:
# Fenêtre de 30 jours = 30 * 24 * 4 = 2880 bougies de 15m
window_30d = 30 * 24 * 4

corr_30d = rolling_correlation(ret_btc, ret_eth, window=window_30d)
plot_rolling_correlation(corr_30d, window=window_30d)

print(f'Corrélation moyenne sur 30 jours : {corr_30d.mean():.3f}')
print(f'Corrélation min : {corr_30d.min():.3f}')
print(f'Corrélation max : {corr_30d.max():.3f}')

## Conclusion Phase 1

**Ce qu'on a établi :**
- [ ] Les données BTC/ETH sont chargées, nettoyées et alignées
- [ ] Nombre de gaps identifiés et compris
- [ ] Les log-returns sont calculés et leur distribution analysée
- [ ] La corrélation mobile BTC/ETH est visualisée

**Ce qu'on verra en Phase 2 :**
Vérifier formellement si BTC et ETH sont **cointégrés** — c'est-à-dire si le spread
$S_t = \text{BTC}_t - \beta \cdot \text{ETH}_t$ est stationnaire.
Une forte corrélation n'implique pas la cointégration, et c'est la cointégration
qui justifie mathématiquement la stratégie mean-reversion.